# PathSnap: HMM & Viterbi Sequential Route Reconstruction
**Course:** UE24CS352A - Machine Learning | PES University  
**Author:** A S Vincent (`vincent2maran@gmail.com`)

This notebook implements:
1. Road network routing distance calculation (Dijkstra algorithm on `nx.DiGraph`).
2. HMM Transition Probability formulation ($d_{\text{road}}$ vs $d_{\text{euclid}}$).
3. HMM Emission Probability combining Gaussian spatial error and ML model confidence.
4. Log-Space Viterbi Dynamic Programming Trellis Decoder.
5. Evaluation of Route Continuity Score (comparing Classifier-Only vs Classifier + HMM).
6. Visualization of the reconstructed vehicle route on Folium map.


In [ ]:
!pip install -q pyproj networkx folium scikit-learn pandas numpy

import numpy as np
import pandas as pd
import networkx as nx
import folium
import matplotlib.pyplot as plt


## 1. HMM Transition and Emission Models


In [ ]:
class TransitionModel:
    def __init__(self, graph, beta=30.0, max_dist=300.0):
        self.graph = graph
        self.beta = beta
        self.max_dist = max_dist
        self.cache = {}

    def log_transition(self, u, v, d_euclid):
        if u == v:
            d_road = 0.0
        elif self.graph.has_edge(u, v):
            d_road = float(self.graph[u][v].get("weight", 0.0))
        else:
            key = (u, v)
            if key in self.cache:
                d_road = self.cache[key]
            else:
                try:
                    d_road, _ = nx.single_source_dijkstra(
                        self.graph, source=u, target=v, cutoff=self.max_dist, weight="weight"
                    )
                except Exception:
                    d_road = self.max_dist
                self.cache[key] = d_road
        return -np.log(self.beta) - (abs(d_road - d_euclid) / self.beta)

class EmissionModel:
    def __init__(self, sigma=12.0, ml_weight=0.5):
        self.sigma = sigma
        self.ml_weight = ml_weight
        self.const = -np.log(self.sigma * np.sqrt(2.0 * np.pi))

    def log_emission(self, dist_perp, ml_prob):
        spatial_log = self.const - (dist_perp**2) / (2.0 * (self.sigma**2))
        ml_log = np.log(np.clip(ml_prob, 1e-4, 1.0 - 1e-4))
        return (1.0 - self.ml_weight) * spatial_log + self.ml_weight * ml_log


## 2. Log-Space Viterbi Dynamic Programming Algorithm


In [ ]:
def viterbi_decode(obs_candidates, trans_model, emiss_model, d_euclids):
    T = len(obs_candidates)
    if T == 0:
        return []

    V = [{}]
    bp = [{}]

    # Step 0
    for i, c in enumerate(obs_candidates[0]):
        V[0][i] = emiss_model.log_emission(c["dist_perp"], c["ml_prob"])
        bp[0][i] = -1

    # Steps 1 to T-1
    for t in range(1, T):
        V.append({})
        bp.append({})
        d_e = d_euclids[t] if t < len(d_euclids) else 30.0

        for j, curr in enumerate(obs_candidates[t]):
            log_e = emiss_model.log_emission(curr["dist_perp"], curr["ml_prob"])
            best_val = -float('inf')
            best_prev = 0

            for i, prev in enumerate(obs_candidates[t-1]):
                log_t = trans_model.log_transition(prev["to_node"], curr["from_node"], d_e)
                val = V[t-1][i] + log_t + log_e
                if val > best_val:
                    best_val = val
                    best_prev = i

            V[t][j] = best_val
            bp[t][j] = best_prev

    # Backtracking
    best_last = max(V[T-1], key=V[T-1].get)
    path = []
    curr = best_last
    for t in range(T-1, -1, -1):
        path.append(obs_candidates[t][curr]["arc_id"])
        curr = bp[t][curr]
    path.reverse()
    return path

print("Viterbi decoder defined successfully!")
